In [2]:
from google.cloud import bigquery
import google.auth

PROJECT_ID = "qwiklabs-gcp-00-f469c11ef7ce"
DATASET    = "aero_alerts"
REGION     = "US"

credentials, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
client = bigquery.Client(project=PROJECT_ID, location=REGION, credentials=credentials)

client.query(
    f"CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{DATASET}` OPTIONS(location='{REGION}')"
).result()
print("Auth OK — dataset ready:", DATASET)

Auth OK — dataset ready: aero_alerts


In [3]:
from google.cloud import storage
import requests

BUCKET_NAME = f"{PROJECT_ID}-aero-alerts"
storage_client = storage.Client(project=PROJECT_ID, credentials=credentials)

# create your own bucket (skip if it already exists)
try:
    storage_client.create_bucket(BUCKET_NAME, location="US")
    print("Created bucket:", BUCKET_NAME)
except Exception:
    print("Bucket already exists:", BUCKET_NAME)

# download the workshop CSV and stage it in YOUR bucket
csv_data = requests.get(
    "https://storage.googleapis.com/labs.roitraining.com/data-to-ai-workshop/airports.csv"
).content
storage_client.bucket(BUCKET_NAME).blob("airports.csv").upload_from_string(
    csv_data, content_type="text/csv"
)

# point the loader at YOUR bucket now
SOURCE_URI = f"gs://{BUCKET_NAME}/airports.csv"
print("Staged in Cloud Storage:", SOURCE_URI)

Created bucket: qwiklabs-gcp-00-f469c11ef7ce-aero-alerts
Staged in Cloud Storage: gs://qwiklabs-gcp-00-f469c11ef7ce-aero-alerts/airports.csv


In [4]:
SOURCE_URI   = f"gs://{BUCKET_NAME}/airports.csv"
AIRPORTS_RAW = f"{PROJECT_ID}.{DATASET}.airports_raw"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)
client.load_table_from_uri(SOURCE_URI, AIRPORTS_RAW, job_config=job_config).result()

table = client.get_table(AIRPORTS_RAW)
print(f"Loaded {table.num_rows:,} rows, {len(table.schema)} columns\n")
for f in table.schema:
    print(f"  {f.name:25s} {f.field_type}")

print("\n--- sample rows ---")
client.query(f"SELECT * FROM `{AIRPORTS_RAW}` LIMIT 5").to_dataframe()

Loaded 82,893 rows, 19 columns

  id                        INTEGER
  ident                     STRING
  type                      STRING
  name                      STRING
  latitude_deg              FLOAT
  longitude_deg             FLOAT
  elevation_ft              INTEGER
  continent                 STRING
  iso_country               STRING
  iso_region                STRING
  municipality              STRING
  scheduled_service         BOOLEAN
  icao_code                 STRING
  iata_code                 STRING
  gps_code                  STRING
  local_code                STRING
  home_link                 STRING
  wikipedia_link            STRING
  keywords                  STRING

--- sample rows ---


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11,NA,US,US-PA,Bensalem,False,None,None,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,None,None
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435,NA,US,US-KS,Leoti,False,None,None,00AA,00AA,None,None,None
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450,NA,US,US-AK,Anchor Point,False,None,None,00AK,00AK,None,None,None
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820,NA,US,US-AL,Harvest,False,None,None,00AL,00AL,None,None,None
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80,NA,US,US-AK,King Salmon,False,None,None,00AN,00AN,None,None,None


In [5]:
LARGE_AIRPORTS = f"{PROJECT_ID}.{DATASET}.large_airports"

client.query(f"""
CREATE OR REPLACE TABLE `{LARGE_AIRPORTS}` AS
SELECT
  ident,
  iata_code,
  name,
  municipality                      AS city,
  SPLIT(iso_region, '-')[OFFSET(1)] AS state,
  latitude_deg                      AS lat,
  longitude_deg                     AS lng
FROM `{AIRPORTS_RAW}`
WHERE type = 'large_airport'
  AND iso_country = 'US'
  AND latitude_deg  IS NOT NULL
  AND longitude_deg IS NOT NULL
ORDER BY name
""").result()

n = list(client.query(f"SELECT COUNT(*) AS n FROM `{LARGE_AIRPORTS}`").result())[0].n
print(f"{n} large US airports")

client.query(f"SELECT * FROM `{LARGE_AIRPORTS}`").to_dataframe()

71 large US airports


,ident,iata_code,name,city,state,lat,lng
0,KABQ,ABQ,Albuquerque International Sunport,Albuquerque,NM,35.039976,-106.608925
1,KAUS,AUS,Austin Bergstrom International Airport,Austin,TX,30.197535,-97.662015
2,KBWI,BWI,Baltimore/Washington International Thurgood Ma...,Baltimore,MD,39.175400,-76.668297
3,KBDL,BDL,Bradley International Airport,Hartford,CT,41.938510,-72.688066
4,KBUF,BUF,Buffalo Niagara International Airport,Buffalo,NY,42.940498,-78.732201
...,...,...,...,...,...,...,...
66,PANC,ANC,Ted Stevens Anchorage International Airport,Anchorage,AK,61.179004,-149.992561
67,KPVD,PVD,Theodore Francis Green State Airport,Warwick,RI,41.725038,-71.425668
68,KTUL,TUL,Tulsa International Airport,Tulsa,OK,36.198399,-95.888100
69,KIAD,IAD,Washington Dulles International Airport,Dulles,VA,38.944500,-77.455803


In [6]:
import requests, time, pandas as pd

USER_AGENT = "AeroAlertsWorkshop/1.0 (Data-to-AI capstone project)"
HEADERS    = {"User-Agent": USER_AGENT, "Accept": "application/geo+json"}
FORECASTS_TABLE = f"{PROJECT_ID}.{DATASET}.airport_forecasts"

airports = client.query(f"SELECT * FROM `{LARGE_AIRPORTS}`").to_dataframe()

def get_forecast(lat, lng):
    # Step 1: points endpoint -> gives us the forecast URL for this grid
    pts = requests.get(f"https://api.weather.gov/points/{lat:.4f},{lng:.4f}",
                       headers=HEADERS, timeout=30)
    pts.raise_for_status()
    forecast_url = pts.json()["properties"]["forecast"]
    # Step 2: fetch the forecast, return the current/next period
    fc = requests.get(forecast_url, headers=HEADERS, timeout=30)
    fc.raise_for_status()
    return fc.json()["properties"]["periods"][0]

rows, errors = [], 0
for _, a in airports.iterrows():
    try:
        p = get_forecast(a["lat"], a["lng"])
        rows.append({
            "ident": a["ident"], "iata_code": a["iata_code"], "name": a["name"],
            "city": a["city"], "state": a["state"],
            "lat": float(a["lat"]), "lng": float(a["lng"]),
            "forecast_period":   p["name"],
            "temperature_f":     p["temperature"],
            "wind":              p["windSpeed"],
            "short_forecast":    p["shortForecast"],
            "detailed_forecast": p["detailedForecast"],
        })
    except Exception as e:
        errors += 1
        print(f"  ! {a['iata_code']} ({a['name']}): {e}")
    time.sleep(0.5)   # be polite to the API

forecasts = pd.DataFrame(rows)
print(f"\nFetched {len(forecasts)} forecasts ({errors} errors)")

client.load_table_from_dataframe(
    forecasts, FORECASTS_TABLE,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")
).result()
print(f"Saved -> {FORECASTS_TABLE}")

forecasts.head()


Fetched 71 forecasts (0 errors)
Saved -> qwiklabs-gcp-00-f469c11ef7ce.aero_alerts.airport_forecasts


,ident,iata_code,name,city,state,lat,lng,forecast_period,temperature_f,wind,short_forecast,detailed_forecast
0,KABQ,ABQ,Albuquerque International Sunport,Albuquerque,NM,35.039976,-106.608925,Today,83,5 to 10 mph,Partly Sunny then Chance Showers And Thunderst...,A chance of showers and thunderstorms after no...
1,KAUS,AUS,Austin Bergstrom International Airport,Austin,TX,30.197535,-97.662015,Today,90,0 to 10 mph,Partly Sunny then Chance Showers And Thunderst...,A chance of showers and thunderstorms after 1p...
2,KBWI,BWI,Baltimore/Washington International Thurgood Ma...,Baltimore,MD,39.175400,-76.668297,Today,83,7 mph,Sunny,"Sunny, with a high near 83. North wind around ..."
3,KBDL,BDL,Bradley International Airport,Hartford,CT,41.938510,-72.688066,Today,84,1 to 5 mph,Sunny,"Sunny, with a high near 84. Northwest wind 1 t..."
4,KBUF,BUF,Buffalo Niagara International Airport,Buffalo,NY,42.940498,-78.732201,Today,78,5 to 12 mph,Sunny,"Sunny, with a high near 78. Southwest wind 5 t..."


In [7]:
CONNECTION = "gemini_conn"
MODEL      = f"{PROJECT_ID}.{DATASET}.gemini_model"

client.query(f"""
CREATE OR REPLACE MODEL `{MODEL}`
REMOTE WITH CONNECTION `{PROJECT_ID}.us.{CONNECTION}`
OPTIONS (endpoint = 'gemini-2.5-flash')
""").result()

print("Gemini model ready:", MODEL)

Gemini model ready: qwiklabs-gcp-00-f469c11ef7ce.aero_alerts.gemini_model


In [8]:
ALERTS_TABLE = f"{PROJECT_ID}.{DATASET}.airport_alerts"

generate_sql = f"""
CREATE OR REPLACE TABLE `{ALERTS_TABLE}` AS
WITH generated AS (
  SELECT
    ident, iata_code, name, city, state, lat, lng,
    temperature_f, wind, short_forecast,
    ml_generate_text_llm_result AS raw
  FROM ML.GENERATE_TEXT(
    MODEL `{MODEL}`,
    (
      SELECT
        *,
        CONCAT(
          'You are an aviation weather communications officer for the FAA Aero Alerts system. ',
          'Write a brief, plain-language travel advisory (2-3 sentences) for passengers and staff at this airport, ',
          'based only on the forecast below. If conditions could disrupt flights (thunderstorms, fog/low visibility, ',
          'high winds, snow or ice, extreme heat), name the hazard and give one practical tip. If conditions are calm, ',
          'give a short reassuring note. Do not invent data.\\n\\n',
          'Respond in EXACTLY this format:\\n',
          'Severity: <Low, Moderate, or High>\\n',
          'Advisory: <your 2-3 sentence advisory>\\n\\n',
          'Airport: ', name, ' (', iata_code, '), ', city, ', ', state, '\\n',
          'Forecast period: ', forecast_period, '\\n',
          'Temperature: ', CAST(temperature_f AS STRING), ' F\\n',
          'Wind: ', wind, '\\n',
          'Summary: ', short_forecast, '\\n',
          'Details: ', detailed_forecast
        ) AS prompt
      FROM `{FORECASTS_TABLE}`
    ),
    STRUCT(0.2 AS temperature, 512 AS max_output_tokens, TRUE AS flatten_json_output)
  )
)
SELECT
  ident, iata_code, name, city, state, lat, lng,
  temperature_f, wind, short_forecast,
  REGEXP_EXTRACT(raw, r'(?i)Severity:\\s*(Low|Moderate|High)') AS severity,
  TRIM(REGEXP_REPLACE(raw, r'(?is)^.*?Advisory:\\s*', ''))     AS alert,
  CURRENT_TIMESTAMP() AS generated_at
FROM generated
"""
client.query(generate_sql).result()
print("Alerts generated ->", ALERTS_TABLE)

client.query(f"""
SELECT iata_code, city, state, temperature_f, severity, alert
FROM `{ALERTS_TABLE}`
ORDER BY CASE severity WHEN 'High' THEN 1 WHEN 'Moderate' THEN 2 ELSE 3 END
LIMIT 10
""").to_dataframe()

Alerts generated -> qwiklabs-gcp-00-f469c11ef7ce.aero_alerts.airport_alerts


,iata_code,city,state,temperature_f,severity,alert
0,MIA,Miami,FL,84,High,Expect widespread showers and thunderstorms th...
1,FLL,Fort Lauderdale,FL,83,High,Expect showers and thunderstorms throughout th...
2,PBI,West Palm Beach,FL,84,High,Expect showers and thunderstorms throughout th...
3,DEN,Denver,CO,84,Moderate,Passengers and staff at Denver International A...
4,LAX,Los Angeles,CA,71,Moderate,Passengers and staff at LAX should be aware of...
5,SFB,Orlando,FL,83,Moderate,Expect mostly cloudy skies and warm temperatur...
6,MSY,New Orleans,LA,82,Moderate,Passengers and staff at MSY should anticipate ...
7,ABQ,Albuquerque,NM,83,Moderate,Passengers and staff at ABQ should be aware of...
8,RSW,Fort Myers,FL,83,Moderate,Southwest Florida International Airport (RSW) ...
9,SFO,San Francisco,CA,71,Moderate,"Expect sunny skies at SFO today, but be aware ..."


In [9]:
client.query(f"""
SELECT severity, COUNT(*) AS n
FROM `{ALERTS_TABLE}`
GROUP BY severity ORDER BY n DESC
""").to_dataframe()

,severity,n
0,Low,50
1,Moderate,18
2,High,3
